In [ ]:
#!/usr/bin/env python3
"""
AI Software Development Team
Production-Grade Multi-Agent System — LangGraph + LangChain + Groq

Agents   : Researcher · Architect · Coder · Reviewer · Tester
Routing  : LLM-powered Supervisor with structured output
HITL     : LangGraph interrupt() for human escalation
Generic  : Works for any software development request — not just OAuth

Usage:
    python ai_dev_team.py
    python ai_dev_team.py "Build a REST API for a task management system"

Dependencies:
    pip install langgraph langchain langchain-groq python-dotenv rich pydantic
"""

# =========================================================
# IMPORTS
# =========================================================

from __future__ import annotations

import functools
import logging
import operator
import os
import sys
from datetime import datetime, timezone
from enum import Enum
from typing import Annotated, Any, Literal, Optional, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from rich.console import Console
from rich.logging import RichHandler

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt


# =========================================================
# LOGGING
# =========================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    datefmt="[%X]",
    handlers=[RichHandler(rich_tracebacks=True, markup=True)],
)

log = logging.getLogger("dev_team")
console = Console()


# =========================================================
# SETTINGS
# All config sourced from environment — no secrets in code.
# =========================================================

class Settings(BaseModel):
    groq_api_key:      str
    model_name:        str   = "groq:llama-3.3-70b-versatile"
    temperature:       float = 0.0
    max_retries:       int   = 3
    langsmith_tracing: bool  = False
    langsmith_project: str   = "ai-dev-team"

    @classmethod
    def from_env(cls) -> Settings:
        load_dotenv()
        return cls(
            groq_api_key      = os.environ["GROQ_API_KEY"],
            model_name        = os.getenv("MODEL_NAME", "groq:llama-3.3-70b-versatile"),
            temperature       = float(os.getenv("TEMPERATURE", "0")),
            max_retries       = int(os.getenv("MAX_RETRIES", "3")),
            langsmith_tracing = os.getenv("LANGSMITH_TRACING", "false").lower() == "true",
            langsmith_project = os.getenv("LANGSMITH_PROJECT", "ai-dev-team"),
        )

    def apply_to_env(self) -> None:
        """Push resolved config into the process environment."""
        os.environ["GROQ_API_KEY"] = self.groq_api_key
        if self.langsmith_tracing:
            os.environ["LANGSMITH_TRACING"]  = "true"
            os.environ["LANGSMITH_PROJECT"]  = self.langsmith_project


# =========================================================
# ENUMS — eliminates magic strings across the codebase
# =========================================================

class AgentName(str, Enum):
    SUPERVISOR = "supervisor"
    RESEARCHER = "researcher"
    ARCHITECT  = "architect"
    CODER      = "coder"
    REVIEWER   = "reviewer"
    TESTER     = "tester"
    HUMAN      = "human"


class RouteStep(str, Enum):
    RESEARCH     = "research"
    ARCHITECT    = "architect"
    CODER        = "coder"
    REVIEW       = "review"
    TESTER       = "tester"
    HUMAN_REVIEW = "human_review"
    FINISH       = "finish"


# =========================================================
# WORKFLOW STATE
#
# Annotated[list, operator.add] → LangGraph appends values
# from each node return instead of replacing the entire list.
# Plain Optional[str] fields are last-write-wins.
# =========================================================

class WorkflowState(TypedDict):

    # ── Core input ─────────────────────────────────────
    user_request: str

    # ── Append-only accumulated fields ─────────────────
    research_notes: Annotated[list[str], operator.add]
    execution_logs: Annotated[list[str], operator.add]
    errors:         Annotated[list[str], operator.add]

    # ── Agent outputs (overwritten each pass) ──────────
    architecture_plan: Optional[str]
    generated_code:    Optional[str]
    review_feedback:   Optional[str]
    test_results:      Optional[str]

    # ── Control flow ───────────────────────────────────
    next_step:       Optional[str]
    review_approved: bool
    retry_count:     int
    max_retries:     int

    # ── Metadata ───────────────────────────────────────
    current_agent: Optional[str]
    status:        str
    started_at:    str


# =========================================================
# TOOLS
# Generic — not tied to any single problem domain.
# Add or swap tools here without touching graph logic.
# =========================================================

@tool
def search_technical_docs(query: str) -> str:
    """
    Search a knowledge base of software engineering best practices.
    Works for any domain: auth, REST APIs, databases, DevOps, testing, etc.
    """
    knowledge_base: dict[str, str] = {
        "oauth": """
OAuth 2.0 Best Practices:
- Use PKCE for public clients
- Validate the state param to prevent CSRF
- Store tokens in HttpOnly cookies, not localStorage
- Short-lived access tokens with refresh token rotation
- Authlib is the recommended Python library
        """,
        "rest api": """
REST API Best Practices:
- Use correct HTTP verbs and status codes
- Version your API (/v1/, /v2/)
- Paginate list endpoints (cursor > offset)
- Rate-limit all public-facing endpoints
- Document with OpenAPI / Swagger
        """,
        "graphql": """
GraphQL Best Practices:
- Use DataLoader to batch N+1 queries
- Implement query depth / complexity limits
- Never expose raw DB errors in responses
- Use persisted queries in production
        """,
        "database": """
Database Best Practices:
- Use connection pooling (SQLAlchemy + asyncpg)
- Always track schema changes via migrations (Alembic)
- Index foreign keys and common filter columns
- Use transactions for multi-step writes
- Separate read replicas at scale
        """,
        "security": """
Security Best Practices:
- Validate and sanitise all input at the boundary
- Use parameterised queries — never raw string SQL
- Store secrets in env vars or a secrets manager
- Audit-log all sensitive operations
- Apply principle of least privilege to all service accounts
        """,
        "caching": """
Caching Best Practices:
- Use Redis for distributed, shared cache
- Set explicit TTLs on every key
- Cache-aside (lazy) pattern for most cases
- Monitor hit/miss ratios in production
- Never cache sensitive user data without encryption
        """,
        "docker": """
Docker Best Practices:
- Multi-stage builds to minimise image size
- Run as non-root user (USER appuser)
- Pin base image versions (FROM python:3.12-slim)
- Add HEALTHCHECK instructions
- Use .dockerignore to exclude dev artefacts
        """,
        "microservices": """
Microservices Best Practices:
- Each service owns its own data store
- Prefer async (event-driven) communication for decoupling
- Implement circuit breakers (tenacity / resilience4j)
- Centralise observability with OpenTelemetry
- Design every endpoint for idempotency
        """,
        "testing": """
Testing Best Practices:
- 80%+ unit coverage on core business logic
- Integration tests for DB, queue, and external services
- Use factories/fixtures — never production data
- Test error paths, not just happy paths
- pytest + pytest-asyncio is the Python standard
        """,
        "websockets": """
WebSocket Best Practices:
- Authenticate the upgrade handshake, not per-message
- Use heartbeats/ping-pong to detect dead connections
- Reconnect with exponential backoff on the client
- Prefer Redis pub/sub for multi-instance fan-out
        """,
        "queue": """
Message Queue Best Practices:
- Design consumers to be idempotent
- Use dead-letter queues for failed messages
- Set appropriate visibility/ack timeouts
- Monitor queue depth as a key SLO metric
- Prefer RabbitMQ for routing flexibility; Kafka for throughput
        """,
    }

    query_lower = query.lower()
    hits = [content for keyword, content in knowledge_base.items()
            if keyword in query_lower]

    if hits:
        return "\n".join(hits)

    return (
        f"No specific docs for '{query}'. "
        "General guidance: follow SOLID principles, write tests, handle errors "
        "gracefully, use structured logging, and keep secrets out of code."
    )


@tool
def generate_architecture_plan(requirement: str, tech_stack: str = "auto") -> str:
    """
    Generate a high-level architecture plan for any software requirement.
    Optionally specify a preferred tech_stack; leave as 'auto' for recommendations.
    """
    stack_note = (
        tech_stack
        if tech_stack.lower() != "auto"
        else "To be determined based on requirement analysis"
    )
    return f"""
## Architecture Plan

### Requirement
{requirement}

### Preferred Stack
{stack_note}

### Layers

| Layer          | Responsibility                                      |
|----------------|-----------------------------------------------------|
| Entry          | API Gateway / Route Handlers / CLI entrypoint       |
| Application    | Use Cases / Orchestration / DTOs                    |
| Domain         | Business Logic / Entities / Domain Events           |
| Infrastructure | DB, Cache, Queue, External API adapters             |
| Cross-cutting  | Auth, Logging, Config, Error Handling, Tracing      |

### Design Principles
- **Separation of Concerns** — each layer has one reason to change
- **Dependency Inversion** — inject adapters, not concrete implementations
- **Fail Fast** — validate input at all boundaries before processing
- **Observability First** — structured logs + distributed traces from day one

### Security Checklist
- [ ] Input validation and sanitisation at every entry point
- [ ] Secrets via environment variables or secrets manager only
- [ ] HTTPS / TLS everywhere in production
- [ ] Least-privilege service accounts
- [ ] Audit log for sensitive operations
- [ ] Rate limiting on all public endpoints
"""


@tool
def lookup_package(package_name: str, ecosystem: str = "python") -> str:
    """
    Return concise, opinionated info about a library or package.
    ecosystem: python | node | java | go | rust
    """
    catalogue: dict[str, dict[str, str]] = {
        "python": {
            "authlib":    "OAuth 2.0 / OIDC. First choice for FastAPI auth flows.",
            "fastapi":    "Async Python web framework. High-performance, type-safe.",
            "flask":      "Synchronous micro-framework. Good for simple APIs.",
            "sqlalchemy": "Industry-standard ORM. Use 2.x async for FastAPI.",
            "alembic":    "Schema migration tool for SQLAlchemy. Required for every DB project.",
            "pydantic":   "Data validation using Python types. Core to FastAPI.",
            "celery":     "Distributed task queue. Pair with Redis or RabbitMQ.",
            "redis":      "redis-py (sync) / redis.asyncio (async). Caching and pub/sub.",
            "pytest":     "Standard Python test framework. Add pytest-asyncio for async.",
            "httpx":      "Async HTTP client. Preferred for testing FastAPI endpoints.",
            "structlog":  "Structured logging. Prefer over stdlib logging in production.",
            "tenacity":   "Retry with exponential backoff. Use for all external calls.",
            "rq":         "Lightweight Redis-backed job queue. Simpler alternative to Celery.",
        },
        "node": {
            "express":  "Minimal, unopinionated web framework.",
            "nestjs":   "Enterprise TypeScript framework. Batteries included.",
            "prisma":   "Type-safe ORM. Excellent DX for TypeScript projects.",
            "zod":      "TypeScript-first schema validation.",
            "passport": "Auth middleware with 500+ strategies.",
            "jest":     "De-facto JS/TS testing framework.",
            "bullmq":   "Redis-backed job queue for Node. Production-ready.",
        },
        "java": {
            "spring-boot":  "Production-grade Java framework. Convention over configuration.",
            "hibernate":    "JPA ORM. Paired with Spring Data.",
            "mapstruct":    "Compile-time DTO mapper — no reflection overhead.",
            "junit5":       "Standard Java testing framework.",
            "resilience4j": "Circuit breaker, retry, rate limiter for Java.",
            "flyway":       "Database migration tool for Java. Prefer over Liquibase for simplicity.",
        },
        "go": {
            "gin":     "Fast HTTP web framework.",
            "gorm":    "ORM for Go. Supports major SQL databases.",
            "testify": "Assertion and mock library for Go tests.",
            "zap":     "High-performance structured logger by Uber.",
            "migrate": "Database migrations library for Go.",
        },
        "rust": {
            "axum":    "Ergonomic, modular async web framework from the Tokio team.",
            "sqlx":    "Async SQL toolkit. Compile-time query checking.",
            "serde":   "Serialisation / deserialisation framework. Ubiquitous.",
            "tokio":   "Async runtime. The standard for Rust async code.",
        },
    }

    eco = catalogue.get(ecosystem.lower(), catalogue["python"])
    info = eco.get(
        package_name.lower(),
        f"No record for '{package_name}'. Check the official documentation.",
    )
    return f"[{ecosystem}] {package_name}: {info}"


@tool
def analyze_code_quality(code: str, language: str = "python") -> str:
    """
    Static-analysis-style review: checks security, error handling,
    structure, and documentation. Returns APPROVED or REJECTED.
    """
    issues:   list[str] = []
    warnings: list[str] = []
    passed:   list[str] = []

    # ── Security ────────────────────────────────────────
    secret_patterns = ["secret_key", "api_key", "password", "token", "secret"]
    for pat in secret_patterns:
        if pat in code.lower():
            line_has_string = any(
                f'{pat} = "' in code.lower() or f"{pat} = '" in code.lower()
                for _ in [None]
            )
            if line_has_string:
                issues.append(
                    f"SECURITY: Possible hardcoded secret ({pat}). "
                    "Load from environment variables or a secrets manager."
                )
                break

    if "%" in code and any(kw in code.lower() for kw in ["select", "insert", "update", "delete"]):
        issues.append(
            "SECURITY: Possible SQL injection via string formatting. "
            "Use parameterised queries."
        )

    # ── Error handling ───────────────────────────────────
    has_error_handling = any(
        kw in code for kw in ["try", "except", "catch", "rescue", "recover"]
    )
    if has_error_handling:
        passed.append("✅ Exception handling present")
    else:
        warnings.append("WARNING: No exception handling found. Add error boundaries.")

    # ── Code structure ───────────────────────────────────
    has_structure = any(
        kw in code for kw in ["def ", "class ", "function ", "func ", "fn "]
    )
    if has_structure:
        passed.append("✅ Structured code (functions / classes)")
    else:
        warnings.append("WARNING: No functions or classes. Avoid raw procedural scripts.")

    # ── Documentation ────────────────────────────────────
    has_docs = any(marker in code for marker in ['"""', "'''", "//", "#", "/*"])
    if has_docs:
        passed.append("✅ Comments / docstrings present")
    else:
        warnings.append("WARNING: No comments or docstrings found.")

    # ── Substantive implementation ────────────────────────
    if len(code.strip()) > 200:
        passed.append("✅ Substantive implementation (>200 chars)")
    else:
        warnings.append("WARNING: Implementation looks minimal — may be incomplete.")

    # ── Verdict ──────────────────────────────────────────
    status = "APPROVED" if not issues else "REJECTED"
    sections = [f"## Code Review\n\n**Status: {status}**"]

    if issues:
        sections.append(
            "### ❌ Issues (blocking)\n"
            + "\n".join(f"- {i}" for i in issues)
        )
    if warnings:
        sections.append(
            "### ⚠️ Warnings\n"
            + "\n".join(f"- {w}" for w in warnings)
        )
    if passed:
        sections.append(
            "### ✅ Passed Checks\n"
            + "\n".join(f"- {p}" for p in passed)
        )

    return "\n\n".join(sections)


@tool
def run_test_suite(code: str, test_type: str = "unit") -> str:
    """
    Simulate running a test suite against a generated implementation.
    test_type: unit | integration | e2e
    Returns a structured PASSED / FAILED report.
    """
    checks: dict[str, bool] = {
        "non_empty_implementation": len(code.strip()) > 100,
        "has_functions_or_classes": any(
            kw in code for kw in ["def ", "class ", "function ", "func "]
        ),
        "has_error_handling": any(
            kw in code for kw in ["try", "except", "catch", "rescue"]
        ),
        "has_return_or_response": "return" in code or "response" in code.lower(),
        "balanced_parentheses":   code.count("(") == code.count(")"),
    }

    passed  = [name for name, ok in checks.items() if ok]
    failed  = [name for name, ok in checks.items() if not ok]
    success = not failed

    return (
        f"## Test Results ({test_type.upper()})\n\n"
        f"**Status: {'ALL TESTS PASSED ✅' if success else f'{len(failed)} TEST(S) FAILED ❌'}**\n\n"
        f"### Passed ({len(passed)}/{len(checks)})\n"
        + ("\n".join(f"  ✅ {t}" for t in passed) or "  (none)")
        + f"\n\n### Failed ({len(failed)}/{len(checks)})\n"
        + ("\n".join(f"  ❌ {t}" for t in failed) or "  (none)")
    )


# =========================================================
# AGENT FACTORY
#
# Builds a ReAct-style agent using raw LangGraph primitives.
# No deprecated create_react_agent / create_agent helpers used.
#
# Flow:
#   ┌──────────┐  tool_calls?  ┌───────────┐
#   │  agent   │ ── Yes ─────► │   tools   │
#   │  (LLM)   │ ◄─ loop ───── │ (ToolNode)│
#   └──────────┘               └───────────┘
#        │ no tool_calls
#        ▼
#       END
# =========================================================

class _AgentMessages(TypedDict):
    messages: Annotated[list, operator.add]


def build_agent(llm: Any, tools: list, system_prompt: str) -> Any:
    """
    Compile a single-purpose ReAct agent sub-graph.

    Args:
        llm:           A LangChain chat model instance.
        tools:         List of @tool-decorated functions.
        system_prompt: Persona and rules injected at the front of every call.

    Returns:
        A compiled LangGraph app (sub-graph) ready for .invoke().
    """
    if not tools:
        raise ValueError("build_agent: at least one tool is required.")

    llm_with_tools = llm.bind_tools(tools)
    tool_node      = ToolNode(tools)

    def _call_llm(state: _AgentMessages) -> dict:
        messages = state["messages"]
        # Prepend system prompt once — only if not already present
        if not any(isinstance(m, SystemMessage) for m in messages):
            messages = [SystemMessage(content=system_prompt)] + messages
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}

    def _route(state: _AgentMessages) -> str:
        last = state["messages"][-1]
        if getattr(last, "tool_calls", None):
            return "tools"
        return END

    g = StateGraph(_AgentMessages)
    g.add_node("agent", _call_llm)
    g.add_node("tools", tool_node)
    g.set_entry_point("agent")
    g.add_conditional_edges("agent", _route, {"tools": "tools", END: END})
    g.add_edge("tools", "agent")
    return g.compile()


def _invoke_agent(agent: Any, prompt: str) -> str:
    """Run a compiled agent sub-graph and return the final text response."""
    result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    return result["messages"][-1].content


# =========================================================
# SYSTEM PROMPTS — generic, domain-agnostic
# =========================================================

_RESEARCH_PROMPT = """
You are a Senior Software Research Engineer.
Given any software requirement, produce concise technical notes covering:
- Recommended technologies and libraries
- Security concerns specific to this domain
- Established best practices
- Common pitfalls to avoid

Rules:
- ALWAYS call a tool before composing your answer. Never rely on memory alone.
- Be concise. No filler. Bullet points preferred.
""".strip()

_ARCHITECT_PROMPT = """
You are a Senior Software Architect.
Given any requirement, design a clean, scalable architecture:
- Identify the right layers and components
- Select appropriate patterns (Clean Arch, CQRS, Event-driven, MVC, etc.)
- Address failure modes — not just the happy path
- Produce a concrete, actionable implementation plan

Rules:
- ALWAYS call a tool first.
- Prefer simplicity. Only introduce complexity where justified.
""".strip()

_CODING_PROMPT = """
You are a Senior Backend Engineer.
Write production-ready code for any requirement:
- Real, runnable code — no TODO placeholders
- Correct imports, type hints, and docstrings
- Comprehensive error handling (try/except with meaningful messages)
- Secrets sourced from environment variables only

Rules:
- ALWAYS call a tool to research packages and best practices first.
- Follow the language's conventions strictly.
- Write code a senior engineer would be proud to review.
""".strip()

_REVIEW_PROMPT = """
You are a Principal Code Reviewer and Security Engineer.
Review code for any language or framework:
- Detect security vulnerabilities (injection, hardcoded secrets, etc.)
- Detect missing error handling
- Verify best practices and code structure

Rules:
- ALWAYS call the analysis tool. Do not review by impression or memory.
- Be strict. Reject code with real issues.
- State APPROVED or REJECTED explicitly at the top of your response.
- Provide specific, actionable feedback for every finding.
""".strip()

_TESTING_PROMPT = """
You are a Senior QA Engineer.
Validate generated implementations for any language:
- Run the appropriate test suite
- Check functional correctness indicators
- Provide a clear PASSED or FAILED verdict with specifics

Rules:
- ALWAYS use the run_test_suite tool first.
- Never guess at results without running the tool.
""".strip()


# =========================================================
# SUPERVISOR — structured output routing
# =========================================================

class SupervisorDecision(BaseModel):
    next_step: Literal[
        "research", "architect", "coder",
        "review", "tester", "human_review", "finish",
    ] = Field(description="The next agent to activate")

    reasoning: str = Field(description="One-line justification for this routing decision")


# =========================================================
# TIMESTAMP HELPER
# =========================================================

def _ts() -> str:
    return datetime.now(timezone.utc).strftime("%H:%M:%S")


# =========================================================
# NODES
# Each node is a pure function: WorkflowState → partial dict.
# Dependencies (llm, agents) are injected via functools.partial.
# =========================================================

def supervisor_node(state: WorkflowState, *, llm: Any) -> dict:
    console.rule("[bold magenta]SUPERVISOR[/bold magenta]")

    structured = llm.with_structured_output(SupervisorDecision)

    prompt = f"""
You are an AI Engineering Manager orchestrating a software dev-team workflow.
Route to the correct agent based on the current state.

User Request      : {state["user_request"]}
Research Done     : {"Yes" if state["research_notes"] else "No"}
Architecture Done : {"Yes" if state["architecture_plan"] else "No"}
Code Generated    : {"Yes" if state["generated_code"] else "No"}
Review Approved   : {state["review_approved"]}
Review Feedback   : {state["review_feedback"] or "None"}
Test Results      : {state["test_results"] or "None"}
Retry Count       : {state["retry_count"]} / {state["max_retries"]}

Routing Rules (evaluate in order):
1. No research notes             → research
2. No architecture plan          → architect
3. No generated code             → coder
4. Code exists, not yet reviewed → review
5. Review rejected + retries left→ coder  (coder will see the feedback)
6. Review approved               → tester
7. Tests complete                → finish
8. Retries exhausted             → human_review
""".strip()

    try:
        decision  = structured.invoke(prompt)
        next_step = decision.next_step

        # Hard safety gate — force escalation when retries are exhausted
        if state["retry_count"] >= state["max_retries"] and not state["review_approved"]:
            next_step = RouteStep.HUMAN_REVIEW.value
            log.warning("Retries exhausted — escalating to human review.")

        log.info(f"[Supervisor] → {next_step} | {decision.reasoning}")

        return {
            "next_step":      next_step,
            "current_agent":  AgentName.SUPERVISOR.value,
            "execution_logs": [f"[{_ts()}] Supervisor → {next_step}: {decision.reasoning}"],
        }

    except Exception as exc:
        log.error(f"[Supervisor] crashed: {exc}")
        return {
            "next_step":      RouteStep.HUMAN_REVIEW.value,
            "errors":         [f"supervisor_node: {exc}"],
            "execution_logs": [f"[{_ts()}] Supervisor error — escalated to human_review"],
        }


def research_node(state: WorkflowState, *, agents: dict) -> dict:
    console.rule("[bold cyan]RESEARCHER[/bold cyan]")
    try:
        output = _invoke_agent(
            agents["researcher"],
            f"Research this requirement:\n\n{state['user_request']}\n\n"
            "Use your tools. Return technologies, security concerns, best practices, and pitfalls.",
        )
        log.info("[Researcher] complete.")
        return {
            "research_notes": [output],
            "current_agent":  AgentName.RESEARCHER.value,
            "execution_logs": [f"[{_ts()}] Research completed"],
        }
    except Exception as exc:
        log.error(f"[Researcher] {exc}")
        return {
            "errors":         [f"research_node: {exc}"],
            "execution_logs": [f"[{_ts()}] Research failed: {exc}"],
        }


def architect_node(state: WorkflowState, *, agents: dict) -> dict:
    console.rule("[bold cyan]ARCHITECT[/bold cyan]")
    try:
        notes  = "\n".join(state["research_notes"])
        output = _invoke_agent(
            agents["architect"],
            f"Design the architecture for this requirement:\n\n{state['user_request']}\n\n"
            f"Research Notes:\n{notes}\n\nUse your tools.",
        )
        log.info("[Architect] complete.")
        return {
            "architecture_plan": output,
            "current_agent":     AgentName.ARCHITECT.value,
            "execution_logs":    [f"[{_ts()}] Architecture completed"],
        }
    except Exception as exc:
        log.error(f"[Architect] {exc}")
        return {
            "errors":         [f"architect_node: {exc}"],
            "execution_logs": [f"[{_ts()}] Architecture failed: {exc}"],
        }


def coding_node(state: WorkflowState, *, agents: dict) -> dict:
    console.rule("[bold cyan]CODER[/bold cyan]")

    feedback_section = (
        f"\nPrevious Review Feedback (fix all issues before resubmitting):\n"
        f"{state['review_feedback']}"
        if state.get("review_feedback")
        else ""
    )

    try:
        notes  = "\n".join(state["research_notes"])
        output = _invoke_agent(
            agents["coder"],
            f"Generate a complete, production-ready implementation.\n\n"
            f"Requirement:\n{state['user_request']}\n\n"
            f"Architecture:\n{state['architecture_plan']}\n\n"
            f"Research:\n{notes}"
            f"{feedback_section}\n\n"
            "Use your tools. Write real, runnable code with imports, type hints, "
            "error handling, and docstrings. No placeholders.",
        )
        log.info(f"[Coder] complete (attempt {state['retry_count'] + 1}).")
        return {
            "generated_code":  output,
            "review_feedback": None,   # clear stale feedback before next review
            "review_approved": False,
            "retry_count":     state["retry_count"] + 1,
            "current_agent":   AgentName.CODER.value,
            "execution_logs":  [f"[{_ts()}] Code generated (attempt {state['retry_count'] + 1})"],
        }
    except Exception as exc:
        log.error(f"[Coder] {exc}")
        return {
            "errors":         [f"coding_node: {exc}"],
            "execution_logs": [f"[{_ts()}] Coding failed: {exc}"],
        }


def review_node(state: WorkflowState, *, agents: dict) -> dict:
    console.rule("[bold cyan]REVIEWER[/bold cyan]")
    try:
        output = _invoke_agent(
            agents["reviewer"],
            f"Review this code carefully. Be strict.\n\n"
            f"{state['generated_code']}\n\n"
            "Use the analysis tool. State APPROVED or REJECTED clearly at the top.",
        )
        # APPROVED is only valid if explicitly present AND REJECTED is absent
        approved = "APPROVED" in output.upper() and "REJECTED" not in output.upper()
        log.info(f"[Reviewer] {'approved ✅' if approved else 'rejected ❌'}")
        return {
            "review_feedback": output,
            "review_approved": approved,
            "current_agent":   AgentName.REVIEWER.value,
            "execution_logs":  [f"[{_ts()}] Review: {'approved' if approved else 'rejected'}"],
        }
    except Exception as exc:
        log.error(f"[Reviewer] {exc}")
        return {
            "errors":         [f"review_node: {exc}"],
            "execution_logs": [f"[{_ts()}] Review failed: {exc}"],
        }


def testing_node(state: WorkflowState, *, agents: dict) -> dict:
    console.rule("[bold cyan]TESTER[/bold cyan]")
    try:
        output = _invoke_agent(
            agents["tester"],
            f"Run tests on this implementation.\n\n"
            f"{state['generated_code']}\n\n"
            "Use the run_test_suite tool. Provide a clear PASSED or FAILED verdict.",
        )
        log.info("[Tester] complete.")
        return {
            "test_results":   output,
            "status":         "SUCCESS",
            "current_agent":  AgentName.TESTER.value,
            "execution_logs": [f"[{_ts()}] Testing completed"],
        }
    except Exception as exc:
        log.error(f"[Tester] {exc}")
        return {
            "errors":         [f"testing_node: {exc}"],
            "status":         "FAILED",
            "execution_logs": [f"[{_ts()}] Testing failed: {exc}"],
        }


def human_review_node(state: WorkflowState) -> dict:
    """
    Human-in-the-loop escalation node.
    Uses LangGraph's interrupt() to pause execution and serialise state.
    The runner resumes with Command(resume=<user_input>).
    """
    console.rule("[bold red]HUMAN REVIEW[/bold red]")

    decision: str = interrupt({
        "reason":          "Automated review failed or max retries exceeded.",
        "retry_count":     state["retry_count"],
        "review_feedback": state.get("review_feedback"),
        "instructions":    "Type 'approve' to approve, or anything else to reject.",
    })

    approved = str(decision).strip().lower() == "approve"
    log.info(f"[Human] {'approved ✅' if approved else 'rejected ❌'}")

    return {
        "review_approved": approved,
        "retry_count":     0,          # reset so coder can retry after human feedback
        "current_agent":   AgentName.HUMAN.value,
        "execution_logs":  [f"[{_ts()}] Human review: {'approved' if approved else 'rejected'}"],
    }


# =========================================================
# CONDITIONAL ROUTER
# =========================================================

_ROUTE_MAP: dict[str, Any] = {
    RouteStep.RESEARCH.value:     "research",
    RouteStep.ARCHITECT.value:    "architect",
    RouteStep.CODER.value:        "coder",
    RouteStep.REVIEW.value:       "review",
    RouteStep.TESTER.value:       "tester",
    RouteStep.HUMAN_REVIEW.value: "human_review",
    RouteStep.FINISH.value:       END,
}


def _route(state: WorkflowState) -> str:
    return state["next_step"]


# =========================================================
# GRAPH BUILDER
# =========================================================

def build_workflow(settings: Settings) -> Any:
    """
    Assemble and compile the full multi-agent LangGraph workflow.

    Design decisions:
    - Each agent is a separate compiled sub-graph (isolation, testability)
    - Nodes receive dependencies via functools.partial (no globals)
    - MemorySaver checkpointer enables interrupt/resume for HITL
    - All agents share one LLM instance (swap per-agent if needed)
    """
    llm = init_chat_model(
        model=settings.model_name,
        temperature=settings.temperature,
    )

    # ── Agent sub-graphs ────────────────────────────────
    agents: dict[str, Any] = {
        "researcher": build_agent(
            llm,
            [search_technical_docs],
            _RESEARCH_PROMPT,
        ),
        "architect": build_agent(
            llm,
            [generate_architecture_plan, lookup_package],
            _ARCHITECT_PROMPT,
        ),
        "coder": build_agent(
            llm,
            [search_technical_docs, lookup_package],
            _CODING_PROMPT,
        ),
        "reviewer": build_agent(
            llm,
            [analyze_code_quality],
            _REVIEW_PROMPT,
        ),
        "tester": build_agent(
            llm,
            [run_test_suite],
            _TESTING_PROMPT,
        ),
    }

    # ── Nodes with injected dependencies ────────────────
    nodes: dict[str, Any] = {
        "supervisor":   functools.partial(supervisor_node, llm=llm),
        "research":     functools.partial(research_node,   agents=agents),
        "architect":    functools.partial(architect_node,  agents=agents),
        "coder":        functools.partial(coding_node,     agents=agents),
        "review":       functools.partial(review_node,     agents=agents),
        "tester":       functools.partial(testing_node,    agents=agents),
        "human_review": human_review_node,
    }

    # ── Graph assembly ───────────────────────────────────
    g = StateGraph(WorkflowState)

    for name, fn in nodes.items():
        g.add_node(name, fn)

    g.set_entry_point("supervisor")

    g.add_conditional_edges("supervisor", _route, _ROUTE_MAP)

    for node in ["research", "architect", "coder", "review", "tester", "human_review"]:
        g.add_edge(node, "supervisor")

    return g.compile(checkpointer=MemorySaver())


# =========================================================
# WORKFLOW RUNNER
# Handles the full interrupt → resume cycle for HITL nodes.
# =========================================================

def run_workflow(user_request: str, max_retries: int = 3) -> WorkflowState:
    """
    Execute the multi-agent dev-team for any software requirement.

    Args:
        user_request: Natural-language description of what to build.
        max_retries:  Max automated code → review cycles before human escalation.

    Returns:
        Final WorkflowState after all agents complete.
    """
    settings = Settings.from_env()
    settings.apply_to_env()

    app    = build_workflow(settings)
    config = {"configurable": {"thread_id": "dev-team-001"}}

    initial: WorkflowState = {
        "user_request":      user_request,
        "research_notes":    [],
        "architecture_plan": None,
        "generated_code":    None,
        "review_feedback":   None,
        "test_results":      None,
        "next_step":         None,
        "review_approved":   False,
        "retry_count":       0,
        "max_retries":       max_retries,
        "current_agent":     None,
        "execution_logs":    [],
        "errors":            [],
        "status":            "RUNNING",
        "started_at":        datetime.now(timezone.utc).isoformat(),
    }

    console.rule("[bold green]AI DEV TEAM — STARTED[/bold green]")
    log.info(f"Request: {user_request[:120]}")

    # ── Primary invoke ────────────────────────────────────
    result = app.invoke(initial, config=config)

    # ── HITL interrupt/resume loop ────────────────────────
    # interrupt() causes invoke() to return with __interrupt__ in the result.
    # We resume by calling invoke() again with Command(resume=<value>).
    while result.get("__interrupt__"):
        payload = result["__interrupt__"][0].value

        console.print("\n[bold red]⏸  WORKFLOW PAUSED — HUMAN REVIEW REQUIRED[/bold red]\n")
        console.print(payload)

        human_input = input(
            f"\n{payload.get('instructions', 'Type approve or reject')}: "
        ).strip()

        result = app.invoke(Command(resume=human_input), config=config)

    # ── Final report ──────────────────────────────────────
    console.rule("[bold green]FINAL REPORT[/bold green]")
    console.print(f"[bold]Status :[/bold] {result.get('status', 'UNKNOWN')}")
    console.print(f"[bold]Retries:[/bold] {result.get('retry_count', 0)}")
    console.print(f"[bold]Errors :[/bold] {len(result.get('errors', []))}")

    if result.get("errors"):
        console.print("\n[bold red]Errors:[/bold red]")
        for err in result["errors"]:
            console.print(f"  • {err}")

    console.print("\n[bold cyan]Execution Log:[/bold cyan]")
    for entry in result.get("execution_logs", []):
        console.print(f"  {entry}")

    return result


# =========================================================
# ENTRY POINT
# Accepts an optional CLI argument as the user request.
# =========================================================

if __name__ == "__main__":
    request = (
        " ".join(sys.argv[1:])
        if len(sys.argv) > 1
        else (
            "Build a FastAPI REST API for a task management system "
            "with JWT authentication, PostgreSQL, and Redis caching."
        )
    )
    run_workflow(user_request=request)